# Trading Agent Population

A population of $N$ heterogeneous agents trades one share at a time through a limit order book (Chiarella and Iori 2002). Each agent mixes three forecasts: **fundamentalist** (the price returns to the fundamental value $F$), **chartist** (the recent trend continues) and **noise**. Nobody sets the price: `LimitOrderBook.jl` matches the orders, and prices, spreads and fat tails emerge from the interaction.

In [ ]:
using LimitOrderBook, Random, Statistics, StatsBase, Plots

## The market

Prices are stored as whole ticks of 0.01, so an order and its cancellation always refer to exactly the same price level. When an agent's price crosses the other side of the book, the code sends a one-share market order: for a one-share order this is exactly what a marketable limit order would do.

In [ ]:
N, T, F = 100, 5000, 100.0   # agents, periods, fundamental value
tick = 0.01
L, tau, kmax, se = 20, 20, 0.01, 0.005   # chartist window, horizon and order life, max mark-up, noise sd

function hit!(ob, live, side, tp, sgn)   # one-share market order
    fills, _ = submit_market_order!(ob, side, 1)
    foreach(o -> delete!(live, o.orderid), fills)   # filled orders must never be cancelled
    isempty(fills) && return
    push!(tp, fills[1].price * tick)
    push!(sgn, side == BUY_ORDER ? 1 : -1)
end

function simulate(; s1 = 1.0, s2 = 1.0, sn = 1.0, seed = 1)
    rng = Xoshiro(seed)
    g1, g2, gn = s1 .* abs.(randn(rng, N)), s2 .* abs.(randn(rng, N)), sn .* abs.(randn(rng, N))
    ob = OrderBook{Int,Int,Int,Int}()
    live = Dict{Int,Tuple{Int,OrderSide,Int}}()   # order id => (expiry, side, price)
    p, spread = fill(F, T), fill(NaN, T)
    tp, sgn = Float64[], Int[]
    for t in 1:T
        for (id, (expiry, side, px)) in collect(live)
            if expiry <= t
                cancel_order!(ob, id, side, px)
                delete!(live, id)
            end
        end
        last = isempty(tp) ? F : tp[end]
        i = rand(rng, 1:N)
        rbar = length(tp) > L ? mean(diff(log.(tp[end-L:end]))) : 0.0
        rhat = (g1[i] * log(F / last) / tau + g2[i] * rbar + gn[i] * se * randn(rng)) / (g1[i] + g2[i] + gn[i])
        phat = last * exp(rhat * tau)
        k = kmax * rand(rng)
        bid, ask = best_bid_ask(ob)
        if phat > last
            px = max(1, round(Int, phat * (1 - k) / tick))
            if !isnothing(ask) && px >= ask
                hit!(ob, live, BUY_ORDER, tp, sgn)
            else
                submit_limit_order!(ob, t, BUY_ORDER, px, 1)
                live[t] = (t + tau, BUY_ORDER, px)
            end
        elseif phat < last
            px = max(1, round(Int, phat * (1 + k) / tick))
            if !isnothing(bid) && px <= bid
                hit!(ob, live, SELL_ORDER, tp, sgn)
            else
                submit_limit_order!(ob, t, SELL_ORDER, px, 1)
                live[t] = (t + tau, SELL_ORDER, px)
            end
        end
        p[t] = isempty(tp) ? F : tp[end]
        bid, ask = best_bid_ask(ob)
        isnothing(bid) || isnothing(ask) || (spread[t] = (ask - bid) * tick)
    end
    return (; p, tp, sgn, spread, ob)
end

## One run

In [ ]:
sim = simulate()
println("Trades: ", length(sim.tp), ", mean spread: ", round(mean(filter(!isnan, sim.spread)); digits = 3))
plot(sim.p; label = "Last Trade Price", xlabel = "Period", ylabel = "Price")
hline!([F]; label = "Fundamental Value", linestyle = :dash)

## Stylized facts

Trade-to-trade returns should show fat tails (positive excess kurtosis), almost no autocorrelation, and positive autocorrelation of absolute returns (volatility clustering).

In [ ]:
r = diff(log.(sim.tp))
println("Excess kurtosis: ", round(kurtosis(r); digits = 2))
lags = 1:20
plot(lags, autocor(r, lags); label = "Returns", marker = :circle, xlabel = "Lag (Trades)", ylabel = "Autocorrelation")
plot!(lags, autocor(abs.(r), lags); label = "Absolute Returns", marker = :circle)
hline!([0]; label = "", color = :gray)

## Fundamentalists against chartists

One run is one random draw. Run the same 20 seeds (so the same agents) with a weak and a strong chartist component and compare the averages.

In [ ]:
metrics(s) = (vol = std(diff(log.(s.tp))), kurt = kurtosis(diff(log.(s.tp))), gap = mean(abs.(s.p .- F)) / F)
sem(v) = std(v) / sqrt(length(v))
seeds = 1:20
a = [metrics(simulate(s2 = 0.5, seed = s)) for s in seeds]
b = [metrics(simulate(s2 = 2.0, seed = s)) for s in seeds]
for k in keys(first(a))
    va, vb = getindex.(a, k), getindex.(b, k)
    println(rpad(string(k), 6), "weak chartists ", round(mean(va); sigdigits = 3), "   strong chartists ", round(mean(vb); sigdigits = 3), "   difference ", round(mean(vb .- va); sigdigits = 3), " +- ", round(sem(vb .- va); sigdigits = 2))
end

## Exercises

1. Set the noise scale `sn = 0` and run again. What happens to the number of trades, and why?
2. Raise the forecast horizon `tau` to 50. Orders now rest longer: how do the spread and volatility change?
3. Plot the mispricing `abs.(sim.p .- F) ./ F` for a chartist-dominated market (`s1 = 0.2, s2 = 2.0`). Do you see bubbles?
4. Read the Price Impact section of the Diagnostics wiki page and compute the response function $R(\ell)$ from `sim.tp` and `sim.sgn`.